# LangChain LLM Result Reference

Developer-facing statements defined in `langchain_core.outputs.llm_result`.

# `LLMResult: BaseModel`

Contains generated outputs and additional provider-specific information from an LLM or chat-model call.

## Fields

```python
generations: list[list[Generation | ChatGeneration | GenerationChunk | ChatGenerationChunk]] # Generated outputs grouped first by input prompt and then by candidate generation
llm_output: dict[str, Any] | None = None # Free-form model provider-specific output
run: list[RunInfo] | None = None # Run metadata for each model input
type: Literal["LLMResult"] = "LLMResult" # Discriminator used exclusively for serialization
```

For an LLM, `generations` normally contains `Generation` objects. For a chat model, it normally contains `ChatGeneration` objects.

`llm_output` is not standardized, and its keys can vary by provider and over time. Prefer standardized information available on `AIMessage` when possible.

## Constructor

```python
LLMResult(
    *,
    generations: list[list[Generation | ChatGeneration | GenerationChunk | ChatGenerationChunk]], # Generated outputs grouped by input and candidate
    llm_output: dict[str, Any] | None = None, # Free-form provider-specific output
    run: list[RunInfo] | None = None, # Run metadata for each input
    type: Literal["LLMResult"] = "LLMResult", # Serialization discriminator
) -> None
```

## Methods

### `flatten`

Separates the outer generation groups into individual `LLMResult` objects.

```python
flatten(
    self,
) -> list[LLMResult] # Results containing one outer generation group each
```

The first returned result keeps the original `llm_output`. For every later result, `llm_output` is deep-copied and its `"token_usage"` value is replaced with an empty dictionary to avoid downstream token double-counting. When the original `llm_output` is `None`, later results also use `None`.

The returned results do not include the original `run` metadata.

### `__eq__`

Compares this result with another `LLMResult`.

```python
__eq__(
    self,
    other: object, # Object to compare
) -> bool # Whether generations and provider output are equal
```

Equality compares only `generations` and `llm_output`; it ignores `run` metadata. Returns `NotImplemented` when `other` is not an `LLMResult`.

## Behaviour

`LLMResult` instances are unhashable.

In [ ]:
from langchain_core.outputs import Generation, LLMResult # Import Generation and LLMResult

generation1 = Generation(text="Paris is the capital of France.") # Create output for the first prompt
generation2 = Generation(text="Tokyo is the capital of Japan.") # Create output for the second prompt

result = LLMResult( # Create one result containing outputs for two prompts
    generations=[[generation1], [generation2]], # Group generations by prompt
    llm_output={"token_usage": {"total_tokens": 12}}, # Store provider-specific information
) # Finish creating the result

print("First prompt output:", result.generations[0][0].text) # Display the first answer
print("Second prompt output:", result.generations[1][0].text) # Display the second answer
print("LLM information:", result.llm_output) # Display provider information

flattened_results = result.flatten() # Split the result into separate LLMResult objects

for number, item in enumerate(flattened_results, start=1): # Visit each flattened result
    print(f"Flattened result {number}:", item.generations[0][0].text) # Display its text
    print("LLM output:", item.llm_output) # Display its provider information